In [23]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
import os
import requests


In [24]:
# tool decorator to create a tool that can be called by the agent
@tool
def get_currency_exchange_rate(from_currency: str, to_currency: str) -> float:
    """
    Get the exchange rate from one currency to another.
    """
    api_key = os.getenv("EXCHANGE_RATE_API_KEY")
    if not api_key:
        raise RuntimeError("Set the EXCHANGE_RATE_API_KEY environment variable before calling this tool.")

    url = f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{from_currency}/{to_currency}"
    response = requests.get(url)
    data = response.json()
    return data

In [25]:
result = get_currency_exchange_rate.invoke({"from_currency": "USD", "to_currency": "INR"})
result

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1794009601,
 'time_last_update_utc': 'Sat, 07 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1794096001,
 'time_next_update_utc': 'Sun, 08 Nov 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.1392}

In [26]:
@tool
def convert_currency(amount: float, exchange_rate: float) -> float:
    """
    Given an amount and an exchange rate, convert the amount to the target currency.
    """
    return amount * exchange_rate

In [27]:
convert_currency.invoke({"amount": 100, "exchange_rate": result["conversion_rate"]})

9413.92

### tool binding

In [28]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)


In [29]:
tools = [get_currency_exchange_rate, convert_currency]
llm_with_tools = llm.bind_tools(tools)


In [30]:
messages = [
    HumanMessage(
        content="""
        Find the USD to INR exchange rate.
        Then use the convert_currency tool to calculate
        how much 100 USD is worth in INR.
        """
    )
]


In [31]:
messages

[HumanMessage(content='\n        Find the USD to INR exchange rate.\n        Then use the convert_currency tool to calculate\n        how much 100 USD is worth in INR.\n        ', additional_kwargs={}, response_metadata={})]

### tool calling

In [32]:
ai_message = llm_with_tools.invoke(messages)

In [33]:
ai_message.tool_calls

[{'name': 'get_currency_exchange_rate',
  'args': {'from_currency': 'USD', 'to_currency': 'INR'},
  'id': 'call_ObLyNZ4U0zn0t7D5QDa8jrBX',
  'type': 'tool_call'}]

In [34]:
messages.append(ai_message)

In [35]:
messages

[HumanMessage(content='\n        Find the USD to INR exchange rate.\n        Then use the convert_currency tool to calculate\n        how much 100 USD is worth in INR.\n        ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 122, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 3.21e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.21e-05, 'upstream_inference_prompt_cost': 1.83e-05, 'upstream_inference_completions_cost': 1.38e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp

In [36]:
for tool_call in ai_message.tool_calls:
   print(tool_call)

{'name': 'get_currency_exchange_rate', 'args': {'from_currency': 'USD', 'to_currency': 'INR'}, 'id': 'call_ObLyNZ4U0zn0t7D5QDa8jrBX', 'type': 'tool_call'}


### tool execution

In [46]:
import json

for tool_call in ai_message.tool_calls:
    if tool_call["name"] == "get_currency_exchange_rate":
        # Execute first tool
        tool_message_1 = get_currency_exchange_rate.invoke(tool_call)
        print(tool_message_1)

        # Extract exchange rate
        exchange_rate = json.loads(tool_message_1.content)["conversion_rate"]

        print(f"Exchange rate: {exchange_rate}")

        # Store first tool result
        messages.append(tool_message_1)

        # Execute second tool using first tool's output
        tool_message_2 = convert_currency.invoke(
            {"amount": 100, "exchange_rate": exchange_rate}
        )

        print(tool_message_2)

        # Store second tool result
        messages.append(tool_message_2)


In [47]:
messages

[HumanMessage(content='\n        Find the USD to INR exchange rate.\n        Then use the convert_currency tool to calculate\n        how much 100 USD is worth in INR.\n        ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 122, 'total_tokens': 145, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 3.21e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 3.21e-05, 'upstream_inference_prompt_cost': 1.83e-05, 'upstream_inference_completions_cost': 1.38e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp

In [49]:
llm_with_tools.invoke(messages).content

'The exchange rate from USD to INR is 94.1392. Therefore, 100 USD is worth approximately 9413.92 INR.'